# Anemometer and Accelerometer Test Analysis


This notebook performs an analysis of accelerometer and anemometer data to evaluate the performance of the system. It retrieves data from multiple sensors, processes it, and generates plots to visualize the power spectral density (PSD) of accelerometer data, air turbulence measurements from the anemometer, and mount errors. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time, TimeDelta

from lsst.daf.butler import Butler
from lsst.summit.utils.efdUtils import makeEfdClient

%matplotlib inline


## Connecting and Fetching Data

In [ ]:
# Create an EFD client instance
client = makeEfdClient()

# Initialise the Butler with the LATISS dataset
butler = Butler('LATISS', collections="LATISS/raw/all")

## Functions for Processing and Organizing Data

In [ ]:
# Just using this in the notebook for now.
def merge_packed_psd(packed_dataframe, base_field, sensor_names):
    """
    Select fields that represent the Power Spectral Density (PSD) of a sensor
    and unpack them into a dataframe with PSD vs frequency.

    Parameters
    ----------
    packed_dataframe : pandas.DataFrame
        Packed data frame containing the desired data.
    base_field : str
        Base field name that will be expanded to query all vector entries.
    sensor_names : str or list
        Name of the sensor(s) of interest.

    Returns
    -------
    result : pandas.DataFrame
        A DataFrame containing the results of the query.
    """
    # Extracting min and max PSD frequencies and number of data points
    min_psd_frequency = packed_dataframe["minPSDFrequency"].iloc[0]
    max_psd_frequency = packed_dataframe["maxPSDFrequency"].iloc[0]
    num_data_points = packed_dataframe["numDataPoints"].iloc[0]

    # Ensuring sensor names are in a list format
    if isinstance(sensor_names, str):
        sensor_names = [sensor_names]

    # Filtering the dataframe to include only the desired sensor names
    packed_dataframe = packed_dataframe.loc[
        packed_dataframe.sensorName.isin(sensor_names)
    ]

    # Select all fields of base field
    packed_fields = [
        k for k in packed_dataframe.keys() if k.startswith(base_field) and k[len(base_field):].isdigit()
    ]
    packed_fields = sorted(
        packed_fields, key=lambda k: int(k[len(base_field):])
    )  # Sort by pack ID

    # Check for consistency between the number of data points and packed fields
    assert num_data_points == len(packed_fields), "Number of packed data points does not match numDataPoints!"
    
    # Creating an empty array for output and calculating the frequency step
    packed_len = len(packed_dataframe)
    output = np.empty(len(packed_fields) * packed_len)
    delta_f = float(max_psd_frequency - min_psd_frequency) / (len(packed_fields) - 1)  # Frequency step

    # Generating frequency labels and filling the output array with PSD values
    columns = []
    for i, field in enumerate(packed_fields):
        columns.append(min_psd_frequency + i * delta_f)
        output[i::len(packed_fields)] = packed_dataframe[field]

    # Reshape the output array into a DataFrame format   
    output = np.reshape(output, (packed_len, len(packed_fields)))

    return pd.DataFrame(data=output, columns=columns, index=packed_dataframe.index)


# This dictionary defines the axis scramble
true_axes = {
    "AuxTel-M1": {"X": "El", "Y": "Az", "Z": "Opt"},
    "AuxTel-M2": {"X": "El", "Y": "Az", "Z": "Opt"},
    "AuxTel-Truss": {"X": "El", "Y": "Opt", "Z": "Az"},
}


## Analysis of PSD, Anemometer, and Mount Data

In [ ]:
psds = []
mount = []

exp_ids = [
    2023031400400, 2023031400410, 2023031400414, 2023031400525,
    2023031400530, 2023031400540
]

# Loop through each exposure ID and perform analysis
for exp_id in exp_ids:
    metadata = butler.get("raw.metadata", detector=0, exposure=exp_id)
    start_time = Time(metadata["DATE-BEG"])
    end_time = Time(metadata["DATE-END"])
    print(start_time, end_time)

    # Create figure for plotting    
    plt.figure(figsize=(8.5, 11))
    plt.subplots_adjust(wspace=0.5, hspace=1.5)
    plt.suptitle(f"Accelerometer / Anemometer / Mount - {exp_id}", fontsize=16)

    # Retrieve accelerometer data
    accel_data = await client.select_time_series(
        "lsst.sal.ESS.accelerometerPSD", ["*"], start_time, end_time
    )

    index_counter = 0  # First PSD in the sequence
    m12_axes = ["X", "Y", "Z"]
    truss_axes = ["X", "Z", "Y"]
    sensors = ["AuxTel-M1", "AuxTel-M2", "AuxTel-Truss"]
    plot_counter = 1
    psd_sum = []

    # Loop through each sensor to plot PSD data
    for sensor in sensors:
        axes = m12_axes if sensor in ["AuxTel-M1", "AuxTel-M2"] else truss_axes
        for axis in axes:
            base_field = f"accelerationPSD{axis}"
            true_axis = true_axes[sensor][axis]

            plt.subplot(5, 3, plot_counter)
            plt.title(f"{sensor} - {true_axis}\n", fontsize=12)

            df = merge_packed_psd(accel_data, base_field, sensor)
            #row = df.iloc[index_counter][2:] 
            row = df.loc[df.index[index_counter], df.columns[2:]]  # We have replaced this line to avoid warning.
            row.plot()
            psd_sum.append(row.sum())

            plt.xlabel("Frequency [Hz]")
            plt.ylabel("PSD [m²/(Hz s⁴)]")
            plt.ylim(0.0, 5.0e-10)
            plot_counter += 1

    psds.append(psd_sum)

    # Retrieve anemometer data
    anemom_start = start_time - TimeDelta(30.0, format="sec")
    anemom_end = start_time + TimeDelta(30.0, format="sec")
    ane = await client.select_time_series(
        "lsst.sal.ESS.airTurbulence", ["*"], anemom_start, anemom_end
    )

    # Extract mean and std deviation for each axis
    ux_mean, ux_std = ane["speed0"].values[0], ane["speedStdDev0"].values[0]
    uy_mean, uy_std = ane["speed1"].values[0], ane["speedStdDev1"].values[0]
    uz_mean, uz_std = ane["speed2"].values[0], ane["speedStdDev2"].values[0]

    # Plot anemom data
    for i, (label, mean, std) in enumerate(
        zip(["UX", "UY", "UZ"], [ux_mean, uy_mean, uz_mean], [ux_std, uy_std, uz_std])
    ):
        plt.subplot(5, 3, plot_counter + i)
        plt.title(f"Anemom-{label}")
        plt.text(0.1, 0.7, f"Mean={mean:.2f}", horizontalalignment="left", verticalalignment="center")
        plt.text(0.1, 0.3, f"Std={std:.2f}", horizontalalignment="left", verticalalignment="center")

    plot_counter += 3

    # Retrieve mount data
    az = await client.select_packed_time_series(
        "lsst.sal.ATMCS.mount_AzEl_Encoders", "azimuthCalculatedAngle", start_time, end_time
    )
    el = await client.select_packed_time_series(
        "lsst.sal.ATMCS.mount_AzEl_Encoders", "elevationCalculatedAngle", start_time, end_time
    )

    # Extract values and calculate time difference from the middle of the interval
    az_vals, el_vals, times = np.array(az.values[:, 0]), np.array(el.values[:, 0]), np.array(az.values[:, 1])
    fit_times = times - times[int(len(times) / 2)]  # Center time variable in the interval

    # Fit polynomials and compute errors
    az_fit, el_fit = np.polyfit(fit_times, az_vals, 4), np.polyfit(fit_times, el_vals, 4)
    az_model, el_model = np.polyval(az_fit, fit_times), np.polyval(el_fit, fit_times)
    
    # Calculate RMS errors
    az_error, el_error = (az_vals - az_model) * 3600, (el_vals - el_model) * 3600
    az_rms, el_rms = np.sqrt(np.mean(az_error**2)), np.sqrt(np.mean(el_error**2))

    # Calculate image RMS errors
    image_az_rms = az_rms * np.cos(el_vals[0] * np.pi / 180.0)
    image_el_rms = el_rms
    mount.append([image_az_rms, image_el_rms])

    # Plot azimuth error
    plt.subplot(5, 3, plot_counter)
    plt.plot(fit_times, az_error, color="red")
    title = f"Azimuth RMS error = {az_rms:.2f} arcseconds\nImage RMS error = {image_az_rms:.2f} arcseconds"
    plt.title(title, fontsize=10)
    plt.ylim(-4.0, 4.0)
    plt.xticks([])
    plt.ylabel("Arcseconds")

    # Plot elevation error
    plt.subplot(5, 3, plot_counter + 1)
    plt.plot(fit_times, el_error, color="green")
    title = f"Elevation RMS error = {el_rms:.2f} arcseconds\nImage RMS error = {image_el_rms:.2f} arcseconds"
    plt.title(title, fontsize=10)
    plt.ylim(-4.0, 4.0)
    plt.xticks([])

## Summary Plot of Power and Mount Errors

In [ ]:
# Now make the summary plot
mount = np.array(mount)
psds = np.array(psds)
fan_setting = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0]

plt.figure(figsize=(8, 8))
plt.subplots_adjust(hspace=0.3)

# First subplot: M2 Accelerometer Total Power
plt.subplot(2, 1, 1)
plt.title("M2 Accelerometer Total Power")
plt.plot(fan_setting, psds[:, 3], label="M2-El")
plt.plot(fan_setting, psds[:, 4], label="M2-Az")
plt.plot(fan_setting, psds[:, 5], label="M2-Opt")
plt.legend()
plt.xlabel("Fan Setting (Hz)")
plt.ylabel("Total Power (m²/s⁴)")

# Second subplot: Mount Errors
plt.subplot(2, 1, 2)
plt.title("Mount Errors")
plt.plot(fan_setting, mount[:, 0], label="Az")
plt.plot(fan_setting, mount[:, 1], label="El")
plt.legend()
plt.xlabel("Fan Setting (Hz)")
plt.ylabel("RMS Error (arcseconds)")
